# Extract training features with Perch v2 (no-DFT ONNX)

In [1]:
import glob, subprocess, sys, os, re, time
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from pathlib import Path
from tqdm.auto import tqdm
try:
    import onnxruntime as ort
    print('onnxruntime', ort.__version__)
except ImportError:
    whls = glob.glob('/kaggle/input/**/onnxruntime-*.whl', recursive=True)
    assert whls, 'onnxruntime missing and no wheel attached'
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',whls[0]], check=True)
    import onnxruntime as ort
    print('onnxruntime from wheel', ort.__version__)

onnxruntime from wheel 1.24.4


In [2]:
DATA_ROOT   = '/kaggle/input/competitions/birdclef-2026'
OUTPUT_DIR  = '/kaggle/working'
OUT_PARQUET = os.path.join(OUTPUT_DIR, 'features_v4_perch_v2_onnx.parquet')

SAMPLE_RATE   = 32_000
WINDOW_SEC    = 5
WINDOW_SAMPLES = SAMPLE_RATE * WINDOW_SEC      
N_WINDOWS     = 3       
ONNX_BATCH    = 192      
MIN_RATING    = 2
DEBUG_MODE    = False
N_DEBUG_ROWS  = 50

onnx_files = sorted(Path('/kaggle/input').rglob('*.onnx'))
onnx_files = [p for p in onnx_files if 'perch' in str(p).lower() and 'dft' in str(p).lower()]
if not onnx_files:
    onnx_files = [p for p in sorted(Path('/kaggle/input').rglob('*.onnx')) if 'perch' in str(p).lower()]
assert onnx_files, 'no-DFT Perch ONNX not found — attach tuckerarrants/perch-v2-no-dft-onnx'
ONNX_PATH = str(onnx_files[0])
print('ONNX:', ONNX_PATH)

ONNX: /kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx


In [3]:
N_CPU = os.cpu_count() or 4
so = ort.SessionOptions()
so.intra_op_num_threads = N_CPU
so.inter_op_num_threads = 1
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess = ort.InferenceSession(ONNX_PATH, sess_options=so, providers=['CPUExecutionProvider'])
IN_NAME = sess.get_inputs()[0].name           # 'inputs'
out_names = [o.name for o in sess.get_outputs()]
EMB_NAME = 'embedding' if 'embedding' in out_names else out_names[0]
print('input:', IN_NAME, '| embedding output:', EMB_NAME, '| all:', out_names)

def perch_embed(batch):
    out = sess.run([EMB_NAME], {IN_NAME: batch.astype(np.float32)})[0]
    return out.astype(np.float32, copy=False)

input: inputs | embedding output: embedding | all: ['embedding', 'spatial_embedding', 'spectrogram', 'label']


In [4]:
def parse_filename(fn):
    m = re.match(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg', fn)
    if m:
        _, site, _, hms = m.groups()
        return site, int(hms[:2])
    return 'unknown', -1

def top_k_energy_windows(y, sr, seg_sec, k=3, hop_sec=2.5):
    target = int(sr * seg_sec)
    if len(y) <= target:
        return [np.pad(y, (0, max(0, target - len(y))), 'constant')], [0.0]
    hop = max(1, int(sr * hop_sec))
    starts = list(range(0, len(y) - target + 1, hop))
    wins = [y[s:s+target] for s in starts]
    en = np.array([np.sqrt(np.mean(w.astype(np.float64)**2)) for w in wins])
    order = np.argsort(-en)[:k]
    return [wins[i] for i in order], [starts[i]/sr for i in order]

def load_audio(path):
    y, sr = sf.read(path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SAMPLE_RATE:
        y = librosa.resample(y, orig_sr=sr, target_sr=SAMPLE_RATE)
    return y.astype(np.float32)

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

train_df = pd.read_csv(os.path.join(DATA_ROOT, 'train.csv'))
train_df = train_df[train_df['rating'] >= MIN_RATING].reset_index(drop=True)
if DEBUG_MODE:
    train_df = train_df.sample(N_DEBUG_ROWS, random_state=42).reset_index(drop=True)
print(f'Clips to process: {len(train_df)}')

_probe = np.zeros((1, WINDOW_SAMPLES), dtype=np.float32)
EMBED_DIM = int(perch_embed(_probe).shape[-1])
print(f'EMBED_DIM = {EMBED_DIM}')

PERCH_COLS = [f'perch_{j}' for j in range(EMBED_DIM)]
schema = pa.schema(
    [
        ('filename',  pa.string()),
        ('start_sec', pa.float64()),
        ('site',      pa.string()),
        ('hour',      pa.int64()),
    ] + [(c, pa.float32()) for c in PERCH_COLS]
)
writer = pq.ParquetWriter(OUT_PARQUET, schema, compression='snappy')
total_rows = 0

buf_audio, buf_meta = [], []

def flush():
    global total_rows
    if not buf_audio:
        return
    arr = np.stack(buf_audio).astype(np.float32, copy=False)
    embs = perch_embed(arr)                                # (B, EMBED_DIM)
    B = embs.shape[0]
    cols = {
        'filename':  pa.array([m['filename']  for m in buf_meta], type=pa.string()),
        'start_sec': pa.array([m['start_sec'] for m in buf_meta], type=pa.float64()),
        'site':      pa.array([m['site']      for m in buf_meta], type=pa.string()),
        'hour':      pa.array([m['hour']      for m in buf_meta], type=pa.int64()),
    }
    for j, c in enumerate(PERCH_COLS):
        cols[c] = pa.array(embs[:, j], type=pa.float32())
    writer.write_table(pa.Table.from_pydict(cols, schema=schema))
    total_rows += B
    buf_audio.clear(); buf_meta.clear()

t0 = time.time()
for _, r in tqdm(train_df.iterrows(), total=len(train_df), miniters=200):
    fn = r['filename']
    fp = os.path.join(DATA_ROOT, 'train_audio', r['primary_label'], fn)
    if not os.path.exists(fp):
        fp = os.path.join(DATA_ROOT, 'train_audio', fn)
    if not os.path.exists(fp):
        continue
    try:
        y = load_audio(fp)
    except Exception as e:
        print('load fail', fn, e); continue
    base = os.path.basename(fn)
    site, hour = parse_filename(base)
    wins, starts = top_k_energy_windows(y, SAMPLE_RATE, WINDOW_SEC, k=N_WINDOWS)
    for w, st in zip(wins, starts):
        if len(w) != WINDOW_SAMPLES:
            w = np.pad(w, (0, WINDOW_SAMPLES - len(w)), 'constant')[:WINDOW_SAMPLES]
        buf_audio.append(w)
        buf_meta.append({'filename': base, 'start_sec': st, 'site': site, 'hour': hour})
        if len(buf_audio) >= ONNX_BATCH:
            flush()
flush()
writer.close()
print(f'\nDone in {(time.time()-t0)/60:.1f} min. Total rows written: {total_rows}')

Clips to process: 22411
EMBED_DIM = 1536


  0%|          | 0/22411 [00:00<?, ?it/s]

In [ ]:
import pyarrow.parquet as pq
pf = pq.ParquetFile(OUT_PARQUET)
print('File          :', OUT_PARQUET, f'({os.path.getsize(OUT_PARQUET)/1e6:.0f} MB)')
print('Rows          :', pf.metadata.num_rows)
print('Columns       :', pf.metadata.num_columns)
print('Row groups    :', pf.num_row_groups)
print('First 6 cols  :', pf.schema_arrow.names[:6])
print('Last 2 cols   :', pf.schema_arrow.names[-2:])
print('\nNext: Save Version -> create a Kaggle Dataset from this parquet,')
print('then point train_svm_perch.ipynb PERCH_FEATURES_PATH at it.')